### author by yangshichen
### 注意：脚本仅供参考，使用前请仔细阅读

In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
pQTL_dir = '/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Result/pQTL/'
pQTL_temp_dir = '/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Data/SMR_pQTL_besd/result_for_besd/'
pQTL_output_dir = '/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Data/SMR_pQTL_besd/besd/'
smr = '/media/scPBMC1_AnalysisDisk1/huangzhuoli/Script_HPC/software_gaoyue/SMR/smr-1.3.1-linux-x86_64/smr-1.3.1'

In [3]:
#update_esi
bim = pd.read_csv('/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Data/Genetics/10.maf01.bim', header=None, sep='\t')
bim.columns = ['chr','variant_id','dis','pos','A1','A2']
bed_df = pd.read_csv('/media/AnalysisDisk2/Yangshichen/0_HIV_RNA/QTL/01.Dynamic/01.Data/01.Genotype/gene_annotation.txt',sep='\t')
bed_df['chr'] = bed_df['chr'].str.replace('chr', '', regex=False)
bed_df = bed_df[bed_df['chr'].str.isdigit()]
bed_df['chr'] = bed_df['chr'].astype(int)

In [5]:
# 读取显著 eGene
eGene = pd.read_csv('/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Result/pQTL_all_lead_perm_qvalue_0.05.csv')
eGene = eGene['phenotype_id'].unique()
    
# 读取 bulk 下所有 parquet 文件
parquet_files = [f for f in os.listdir(f"{pQTL_dir}") if f.endswith('.parquet')]

pQTL_match = pd.DataFrame()
for file in parquet_files:
    file_path = os.path.join(f"{pQTL_dir}", file)
    pQTL_result = pd.read_parquet(file_path)
    pQTL_result_match = pQTL_result[pQTL_result['phenotype_id'].isin(eGene)]
    pQTL_match = pd.concat([pQTL_match, pQTL_result_match])
    
# make besd 文件
pQTL_match_besd = pQTL_match.loc[:, ['variant_id','phenotype_id','slope','slope','pval_nominal','pval_nominal']]
pQTL_match_besd.columns = ['SNP','gene','beta','t-stat','p-value','FDR']
pQTL_match_besd['t-stat'] = 'NA'
pQTL_match_besd['FDR'] = 'NA'

pqtl_txt = f"{pQTL_temp_dir}/bulk_pQTL_for_besd.txt"
pQTL_match_besd.to_csv(pqtl_txt, index=False, sep='\t')

# Linux command：注意加引号
gi = f'{smr} --eqtl-summary "{pqtl_txt}" --matrix-eqtl-format --make-besd --out "{pQTL_output_dir}bulk"'
os.system(gi)
    

# 更新 .esi 文件
esi_path = f"{pQTL_output_dir}bulk.esi"
esi = pd.read_csv(esi_path, sep='\t', header=None)
esi.columns = ['chr','variant_id','dis','pos','A1','A2','af']
esi = esi[['variant_id']]
esi = pd.merge(esi, bim.loc[:, ['chr','variant_id','dis','pos','A1','A2']], on='variant_id')

af_df = pQTL_match.loc[:, ['variant_id','af']]
af_df = af_df.loc[~af_df.duplicated(subset='variant_id', keep='first')]
esi = pd.merge(esi, af_df, on='variant_id')
esi = esi.loc[:, ['chr','variant_id','dis','pos','A1','A2','af']]

esi_update_path = f"{pQTL_output_dir}bulk_update.esi"
esi.to_csv(esi_update_path, sep='\t', header=None, index=False)

gi = f'{smr} --beqtl-summary "{pQTL_output_dir}bulk" --update-esi "{esi_update_path}"'
os.system(gi)

# 更新 .epi 文件
epi_path = f"{pQTL_output_dir}bulk.epi"
epi = pd.read_csv(epi_path, sep='\t', header=None)
epi.columns = ['chr','prob','dis','pos','gene_id','strand']
epi[['gene_id']] = epi[['prob']]
epi = epi[['prob','gene_id']]
epi = pd.merge(epi, bed_df, on='gene_id')
epi['dis'] = 0
epi['strand'] = '+'
epi = epi[['chr', 'prob', 'dis', 'left', 'gene_id', 'strand']]

epi_update_path = f"{pQTL_output_dir}bulk_update.epi"
epi.to_csv(epi_update_path, index=False, header=None, sep='\t')

gi = f'{smr} --beqtl-summary "{pQTL_output_dir}bulk" --update-epi "{epi_update_path}"'
os.system(gi)

*******************************************************************
* Summary-data-based Mendelian Randomization (SMR)
* Version 1.3.1
* Build at Sep 21 2022 12:13:19, by GCC 8.3
* (C) 2015 Futao Zhang, Zhihong Zhu and Jian Yang
* The University of Queensland
* MIT License
*******************************************************************
Analysis started: 9:57:37,Sat Feb 28,2026

Options:
--eqtl-summary /media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Data/SMR_pQTL_besd/result_for_besd//bulk_pQTL_for_besd.txt
--matrix-eqtl-format 
--make-besd 
--out /media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Data/SMR_pQTL_besd/besd/bulk

Reading eQTL summary data from /media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Data/SMR_pQTL_besd/result_for_besd//bulk_pQTL_for_besd.txt ...
103329 rows to be included from /media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Data/SMR_pQTL_besd/result_for_besd//bulk_pQTL_for_besd.txt.

Generating the .epi file...
29 probes have been saved in the file /media/AnalysisDisk2

0